# 04 · Đánh giá mô hình và phân tích sai số

Notebook này **chỉ đọc artifact đánh giá thật** do `scripts/evaluate.py` tạo ra. Nếu chưa có run hợp lệ, notebook báo thiếu dữ liệu và không dựng số liệu thay thế. Model selection chỉ dùng validation; final test chỉ được mở sau khi selection manifest đã khóa.

## Kiến trúc pipeline

![Evaluation pipeline](images/fig1_evaluation_pipeline_architecture.jpg)

Sơ đồ này là tài liệu kiến trúc, không phải figure thực nghiệm và không được ghi vào `report/figure_manifest.csv`.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'src').exists(), 'Không tìm thấy project root'
print(f'Project root: {PROJECT_ROOT}')

## 1. Nạp các evaluation artifact

Có thể đặt biến môi trường `JENA_EVALUATION_DIRS` thành danh sách thư mục, phân cách bằng `;`. Nếu không đặt, notebook tìm `runs/*/evaluation/metrics.json`. Mỗi thư mục phải có `metrics.json` và `metrics_per_horizon.csv`.

In [ ]:
configured = [Path(item) for item in os.getenv('JENA_EVALUATION_DIRS', '').split(';') if item]
metric_paths = [path / 'metrics.json' for path in configured]
if not metric_paths:
    metric_paths = sorted((PROJECT_ROOT / 'runs').glob('*/evaluation/metrics.json'))

required = {
    'run_id', 'model_name', 'model_version', 'split', 'population_id',
    'checkpoint', 'resolved_config', 'schema_hash', 'scaler_sha256',
    'overall', 'sample_count', 'horizon', 'unit'
}
records = []
for path in metric_paths:
    payload = json.loads(path.read_text(encoding='utf-8'))
    missing = required - set(payload)
    if missing:
        raise ValueError(f'{path} thiếu trường provenance: {sorted(missing)}')
    if payload['unit'] != 'degC' or payload['horizon'] != 72:
        raise ValueError(f'{path} không đúng đơn vị °C hoặc horizon 72')
    records.append({
        'evaluation_dir': path.parent,
        'run_id': payload['run_id'],
        'model_name': payload['model_name'],
        'model_version': payload['model_version'],
        'split': payload['split'],
        'population_id': payload['population_id'],
        'sample_count': payload['sample_count'],
        'mae_degC': payload['overall']['mae'],
        'mse_degC2': payload['overall']['mse'],
        'rmse_degC': payload['overall']['rmse'],
        'baseline_rmse_degC': (payload.get('baseline') or {}).get('rmse'),
        'checkpoint': payload['checkpoint'],
        'schema_hash': payload['schema_hash'],
        'scaler_sha256': payload['scaler_sha256'],
    })

summary = pd.DataFrame(records)
if summary.empty:
    print('CHƯA CÓ ARTIFACT THẬT: chạy scripts/evaluate.py trên prediction của các run validation.')
else:
    if summary['split'].nunique() != 1 or summary['split'].iloc[0] != 'validation':
        raise ValueError('Notebook so sánh model chỉ chấp nhận validation artifacts')
    for column in ('population_id', 'sample_count', 'schema_hash', 'scaler_sha256'):
        if summary[column].nunique() != 1:
            raise ValueError(f'Fair-comparison mismatch: {column}')
    display(summary.drop(columns=['evaluation_dir']).sort_values('rmse_degC'))

## 2. So sánh metric tổng thể và theo horizon

Các biểu đồ dưới đây chỉ xuất hiện khi có artifact thật, cùng validation population, schema và scaler.

In [ ]:
if not summary.empty:
    ordered = summary.sort_values('rmse_degC')
    ax = ordered.set_index('model_name')[['mae_degC', 'rmse_degC']].plot.bar(
        figsize=(10, 5), rot=10, ylabel='Sai số (°C)',
        title=f"Validation — population {ordered['population_id'].iloc[0]}"
    )
    ax.set_xlabel('Model')
    ax.set_ylim(bottom=0)
    ax.legend(['MAE', 'RMSE'])
    plt.tight_layout()
    plt.show()

In [ ]:
if not summary.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    for row in records:
        path = row['evaluation_dir'] / 'metrics_per_horizon.csv'
        horizon = pd.read_csv(path)
        if list(horizon['horizon']) != list(range(1, 73)):
            raise ValueError(f'{path} không chứa đúng 72 horizon liên tiếp')
        ax.plot(horizon['horizon'], horizon['rmse'], label=f"{row['model_name']} ({row['run_id']})")
    ax.set(title='Validation RMSE theo forecast horizon', xlabel='Forecast horizon (giờ)', ylabel='RMSE (°C)', xlim=(1, 72))
    ax.legend()
    plt.tight_layout()
    plt.show()

## 3. Error analysis theo giờ, mùa và worst cases

Các bảng chỉ được đọc nếu `scripts/evaluate.py` đã tạo chúng từ timestamp hợp lệ. Không suy diễn cơ chế thời tiết hoặc attention nếu artifact không chứng minh được.

In [ ]:
if not summary.empty:
    selected = summary.sort_values('rmse_degC').iloc[0]
    evaluation_dir = selected['evaluation_dir']
    for filename in ('error_by_hour.csv', 'error_by_season.csv', 'worst_cases.csv'):
        path = evaluation_dir / filename
        print(f'\n{filename} — run {selected["run_id"]}')
        if path.exists():
            display(pd.read_csv(path).head(20))
        else:
            print('Không có bảng này; timestamp có thể chưa đủ semantics để phân nhóm.')

## 4. Quyết định và bằng chứng cần bàn giao

- Chạy `scripts/compare_models.py` để áp dụng gate cải thiện RMSE tối thiểu 3% so với persistence baseline và tạo `selection_manifest.json` bất biến.
- Không ghi tên model thắng cuộc trong notebook trước khi có manifest thật.
- Chỉ candidate đã khóa được chạy final test một lần; kết quả test không được dùng để quay lại chọn model.
- Figure báo cáo phải được tạo lại từ artifact thật và đăng ký bằng run/model/split trong `report/figure_manifest.csv`.